# 전처리 방식 선택 및 모델 학습
* 전처리 객체는 함수로 분리하여 버전별로 관리 -> feature.py
* 대신 함수별로 선택한 특성은 여기에 기록

### 특성 선택
* Sex
* Pclass(숫자형처럼 생각하는게 더 나을듯 왜냐하면 1,2,3순서 자체에도 정보가 포함됨) -> Fare와 추이가 비슷하기 때문에 Pclass하나로 대체하는게 더 나을듯
* Sex + cat_age(숫자형) -> 그러나 나이대의 그룹을 나누는 기준은 다시 생각해봐야함 (깎아봐야앎)
* cat_parch(숫자형) -> 결측치 처리 -> 있냐 없냐 수준의 범주형으로 변경
* has_cabin(범주형) -> cabin에 대해 결측치 처리, 이진범주로 변경 해야함
* Title(범주형)


-> 근데? 일단은 베이스라인을 먼저 만들기 위해 연관이 약한 특성 제외 특성 가공을 빼고 일반 특성만 파이프라인에 집어넣어보자

In [1]:
import pandas as pd
import joblib

train = pd.read_csv("../data/processed/01/train.csv")
labels_origin = train["Survived"].copy()
train_origin = train.drop("Survived", axis = 1).copy()

## 1. Baseline

In [2]:
from src.feature import prep_baseline
prep = prep_baseline()

### 로지스틱 회귀

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score


train = train_origin.copy()
labels = labels_origin.copy()


baseline_logistic = make_pipeline(prep, LogisticRegression())
baseline_logistic.fit(train, labels)
predict = baseline_logistic.predict(train)


print("학습세트에대한 예측 점수")
accuracy = accuracy_score(labels, predict)
print(accuracy)


print("k-fold 점수")
accuracy = cross_val_score(baseline_logistic, train, labels, cv=5, scoring="accuracy")
print(accuracy.mean())

학습세트에대한 예측 점수
0.7921348314606742
k-fold 점수
0.7893430513148824


### 랜덤 포레스트

In [5]:
from sklearn.ensemble import RandomForestClassifier

train = train_origin.copy()
labels = labels_origin.copy()
baseline_randomforest = make_pipeline(prep, RandomForestClassifier(random_state=42))
baseline_randomforest.fit(train, labels)
predict = baseline_randomforest.predict(train)

print("학습세트에대한 예측 점수")
accuracy = accuracy_score(labels, predict)
print(accuracy)


print("k-fold 점수")
accuracy = cross_val_score(baseline_randomforest, train, labels, cv=5, scoring="accuracy")
print(accuracy.mean())

학습세트에대한 예측 점수
0.9044943820224719
k-fold 점수
0.7893233527036344


사용된 모델 저장

In [ ]:
joblib.dump(pipeline, "../data/model/baseline_logistic_regression.pkl")

In [ ]:
joblib.dump(pipeline, "../data/model/baseline_random_forest.pkl")

### 결과 분석

사용특성
* Age, Parch, Pclass -> 숫자형, 각 수치에 대해 스케일링 적용, 결측치 처리는 기본 중간값 사용
* Sex -> one-hot 인코딩 적용

로지스틱 회귀
* 학습세트 예측 점수 -> 0.7921
* k-fold 평균 점수 -> 0.789343

랜덤 포레스트
* 학습세트에 대한 예측점수 -> 0.9044
    * 이건 애초에 학습세트로 훈련을 시켜놓고 학습세트를 검증셋으로 사용하다보니 결정트리 자체가 과적합돼서 그런것 결정트리의 특성때문임
* k-fold 평균점수 -> 0.789323

미세하게 랜덤 포레스트가 낫다.

## 2. 번외 - 특성 성별 하나만 넣어보기

In [6]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
sex = train_origin[["Sex"]].copy()
model = make_pipeline(OneHotEncoder(), LogisticRegression())
model.fit(sex, labels)
predict = model.predict(sex)
score = accuracy_score(labels, predict)
print("학습셋으로 예측한 점수")
print(score)


score = cross_val_score(model, sex, labels, cv=5, scoring="accuracy")
print("성별 특성만 사용하여 학습한 모델의 K-fold점수")
print(score.mean())
#
# submission = pd.DataFrame({
#     "PassengerId": test["PassengerId"].astype(int),
#     "Survived": predict.astype(int)
# })
# submission = submission.sort_values("PassengerId")
# submission.to_csv("../data/submission/logistic_only_sex.csv", index=False)
#
# print(submission.shape)
# print(submission.head())


학습셋으로 예측한 점수
0.7921348314606742
성별 특성만 사용하여 학습한 모델의 K-fold점수
0.7920713089727174


### 결과 분석
* 오히려 성별만 넣어서 학습 시킨게 특성 여러개를 넣은 것 보다 k-fold점수가 더 높다
* 성별에 따른 생존률은 female    0.764000 / male      0.192641 이다.
* 따라서 단순 성별(여자인 것)만 보고 생존이라고 했을때 맞을 확률이 0.76인것임..